In [1]:
import json 

# Load baselines
baselines_dict = json.load(open('pitchbook_baselines.json'))
print(baselines_dict)

{'IsUSBased': {'5': [{'alpha': 3.0, 'beta': 4.0}, {'alpha': 3.0, 'beta': 4.0}, {'alpha': 4.0, 'beta': 3.0}, {'alpha': 3.0, 'beta': 4.0}, {'alpha': 3.0, 'beta': 4.0}, {'alpha': 4.0, 'beta': 3.0}, {'alpha': 4.0, 'beta': 3.0}, {'alpha': 2.0, 'beta': 5.0}, {'alpha': 3.0, 'beta': 4.0}, {'alpha': 3.0, 'beta': 4.0}, {'alpha': 4.0, 'beta': 3.0}, {'alpha': 2.0, 'beta': 5.0}, {'alpha': 3.0, 'beta': 4.0}, {'alpha': 2.0, 'beta': 5.0}, {'alpha': 4.0, 'beta': 3.0}, {'alpha': 3.0, 'beta': 4.0}, {'alpha': 2.0, 'beta': 5.0}, {'alpha': 2.0, 'beta': 5.0}, {'alpha': 3.0, 'beta': 4.0}, {'alpha': 3.0, 'beta': 4.0}, {'alpha': 2.0, 'beta': 5.0}, {'alpha': 4.0, 'beta': 3.0}, {'alpha': 1.0, 'beta': 6.0}, {'alpha': 2.0, 'beta': 5.0}, {'alpha': 4.0, 'beta': 3.0}], '10': [{'alpha': 4.0, 'beta': 8.0}, {'alpha': 3.0, 'beta': 9.0}, {'alpha': 7.0, 'beta': 5.0}, {'alpha': 6.0, 'beta': 6.0}, {'alpha': 7.0, 'beta': 5.0}, {'alpha': 2.0, 'beta': 10.0}, {'alpha': 6.0, 'beta': 6.0}, {'alpha': 6.0, 'beta': 6.0}, {'alpha': 7.0

In [ ]:
import pandas as pd 

for var, baseline_info in baselines_dict.items():
    print(f"\n{'='*60}")
    print(f"Variable: {var}")
    print(f"{'='*60}")
    samples_path = 'pitchbook/baseline_data_samples/{}'.format(var)

    # Compute outlier thresholds using all baseline data samples for this variable (IQR method)
    all_samples_path = samples_path + '/nALL_trial0.csv'
    all_samples_df = pd.read_csv(all_samples_path)
    cols = list(all_samples_df.columns) 
    
    # Get statistics from ALL data
    all_data = all_samples_df[cols[0]]
    q1 = all_data.quantile(0.25)
    q3 = all_data.quantile(0.75)
    iqr = q3 - q1
    lower_outlier_threshold = q1 - 1.5 * iqr
    upper_outlier_threshold = q3 + 1.5 * iqr

    if all_data.min() == 0 and all_data.max() == 1: 
        continue

    print(f"\nALL Data Statistics:")
    print(f"  Min: {all_data.min():.2f}")
    print(f"  Q1:  {q1:.2f}")
    print(f"  Q3:  {q3:.2f}")
    print(f"  Max: {all_data.max():.2f}")
    print(f"  Lower Outlier Threshold: {lower_outlier_threshold:.2f}")
    print(f"  Upper Outlier Threshold: {upper_outlier_threshold:.2f}")
    print(f"  Total samples in ALL: {len(all_data)}")

    sample_size_to_outlier_count = {}
    for sample_size, sample_size_baselines in baseline_info.items():
        sample_size = sample_size.replace('_lognorm', '')
        sample_size_to_outlier_count[sample_size] = []
        
        print(f"\n  Sample size: {sample_size}")
        for idx, trial in enumerate(sample_size_baselines):
            trial_samples_path = samples_path + '/n{}_trial{}.csv'.format(sample_size, idx)
            trial_samples_df = pd.read_csv(trial_samples_path)
            trial_data = trial_samples_df[cols[0]]

            # Check for outliers in this trial's samples
            outlier_mask_lower = trial_data < lower_outlier_threshold
            outlier_mask_upper = trial_data > upper_outlier_threshold
            outlier_mask = outlier_mask_lower | outlier_mask_upper

            num_outliers = outlier_mask.sum()
            total_samples = len(trial_data)
            outlier_pct = 100.0 * num_outliers / total_samples
            
            if num_outliers > 0:
                outlier_values = trial_data[outlier_mask].values
                print(f"    Trial {idx}: {num_outliers}/{total_samples} outliers ({outlier_pct:.1f}%) - values: {outlier_values}")
            else:
                print(f"    Trial {idx}: {num_outliers}/{total_samples} outliers ({outlier_pct:.1f}%)")
            
            sample_size_to_outlier_count[sample_size].append(float(num_outliers) / float(total_samples))

    print(f"\n  Sample Size to Outlier Proportions:")
    for size, proportions in sample_size_to_outlier_count.items():
        avg_pct = 100.0 * sum(proportions) / len(proportions) if proportions else 0
        print(f"    {size}: avg {avg_pct:.2f}% outliers across {len(proportions)} trials")
    
    


Variable: IsUSBased

ALL Data Statistics:
  Min: 0.00
  Q1:  0.00
  Q3:  1.00
  Max: 1.00
  Lower Outlier Threshold: -1.50
  Upper Outlier Threshold: 2.50
  Total samples in ALL: 70697

  Sample size: 5
    Trial 0: 0/5 outliers (0.0%)
    Trial 1: 0/5 outliers (0.0%)
    Trial 2: 0/5 outliers (0.0%)
    Trial 3: 0/5 outliers (0.0%)
    Trial 4: 0/5 outliers (0.0%)
    Trial 5: 0/5 outliers (0.0%)
    Trial 6: 0/5 outliers (0.0%)
    Trial 7: 0/5 outliers (0.0%)
    Trial 8: 0/5 outliers (0.0%)
    Trial 9: 0/5 outliers (0.0%)
    Trial 10: 0/5 outliers (0.0%)
    Trial 11: 0/5 outliers (0.0%)
    Trial 12: 0/5 outliers (0.0%)
    Trial 13: 0/5 outliers (0.0%)
    Trial 14: 0/5 outliers (0.0%)
    Trial 15: 0/5 outliers (0.0%)
    Trial 16: 0/5 outliers (0.0%)
    Trial 17: 0/5 outliers (0.0%)
    Trial 18: 0/5 outliers (0.0%)
    Trial 19: 0/5 outliers (0.0%)
    Trial 20: 0/5 outliers (0.0%)
    Trial 21: 0/5 outliers (0.0%)
    Trial 22: 0/5 outliers (0.0%)
    Trial 23: 0/5 outlie